In [1]:
import pandas as pd
from sqlalchemy import create_engine, text

engine = create_engine(
    "postgresql+psycopg://husein@localhost:5432/revenueiq"
)

with engine.connect() as connection:
    dataset = pd.read_sql(
        text("""
            SELECT *
            FROM analytics.modeling_dataset
            ORDER BY cutoff_date, customer_id
        """),
        connection,
    )

In [2]:
display(dataset.groupby("dataset_split").size())

assert len(dataset) == 8780

assert not dataset.duplicated(
    subset=["customer_id", "cutoff_date"]
).any()

assert dataset["inactive_next_90_days"].notna().all()

assert dataset["inactive_next_90_days"].isin([0, 1]).all()

print("Dataset checks passed.")

dataset_split
test          2772
train         3349
validation    2659
dtype: int64

Dataset checks passed.


In [3]:
train = dataset.loc[
    dataset["dataset_split"] == "train"
].copy()

validation = dataset.loc[
    dataset["dataset_split"] == "validation"
].copy()

test = dataset.loc[
    dataset["dataset_split"] == "test"
].copy()

print("Train:", len(train))
print("Validation:", len(validation))
print("Test:", len(test))

Train: 3349
Validation: 2659
Test: 2772


In [4]:
feature_columns = [
    "recency_days",
    "order_count",
    "purchase_value",
    "avg_order_value",
    "observed_tenure_days",
    "orders_last_90_days",
    "purchase_value_last_90_days",
    "orders_previous_90_days",
]

target_column = 'inactive_next_90_days'

X_train = train[feature_columns].astype('float64')
y_train = train[target_column].astype('int64')

X_test = test[feature_columns].astype('float64')
y_test = test[target_column].astype('int64')

X_validation = validation[feature_columns].astype("float64")
y_validation = validation[target_column].astype("int64")

In [5]:
import numpy as np

assert np.isfinite(X_train.to_numpy()).all()
assert np.isfinite(X_validation.to_numpy()).all()

assert set(y_train.unique()) == {0, 1}
assert set(y_validation.unique()) == {0, 1}

In [8]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
)

In [9]:
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)

,"strategy strategy: {""most_frequent"", ""prior"", ""stratified"", ""uniform"", ""constant""}, default=""prior""Strategy to use to generate predictions.* ""most_frequent"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit`. The `predict_proba` method returns the matching one-hot encoded vector.* ""prior"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit` (like ""most_frequent""). ``predict_proba`` always returns the empirical class distribution of `y` also known as the empirical class prior distribution.* ""stratified"": the `predict_proba` method randomly samples one-hot vectors from a multinomial distribution parametrized by the empirical class prior probabilities. The `predict` method returns the class label which got probability one in the one-hot vector of `predict_proba`. Each sampled row of both methods is therefore independent and identically distributed.* ""uniform"": generates predictions uniformly at random from the list of unique classes observed in `y`, i.e. each class has equal probability.* ""constant"": always predicts a constant label that is provided by the user. This is useful for metrics that evaluate a non-majority class. .. versionchanged:: 0.24 The default value of `strategy` has changed to ""prior"" in version 0.24.",'most_frequent'
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness to generate the predictions when``strategy='stratified'`` or ``strategy='uniform'``.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",None
,"constant constant: int or str or array-like of shape (n_outputs,), default=NoneThe explicit constant as predicted by the ""constant"" strategy. Thisparameter is useful only for the ""constant"" strategy.",None
Name,Type,Value
"class_prior_ class_prior_: ndarray of shape (n_classes,) or list of such arraysFrequency of each class observed in `y`. For multioutput classificationproblems, this is computed independently for each output.","ndarray[float64](2,)","[0.42,0.58]"
"classes_ classes_: ndarray of shape (n_classes,) or list of such arraysUnique class labels observed in `y`. For multi-output classificationproblems, this attribute is a list of arrays as each output has anindependent set of possible classes.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X` hasfeature names that are all strings.","ndarray[object](8,)","['recency_days','order_count','purchase_value',...,'orders_last_90_days', 'purchase_value_last_90_days','orders_previous_90_days']"
n_classes_ n_classes_: int or list of intNumber of label for each output.,int,2
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`.,int,8
n_outputs_ n_outputs_: intNumber of outputs.,int,1
sparse_output_ sparse_output_: boolTrue if the array returned from predict is to be in sparse CSC format.Is automatically set to True if the input `y` is passed in sparseformat.,bool,False


In [12]:
baseline_predictions = baseline.predict(X_validation)
print("Validation accuracy:",
      round(accuracy_score(y_validation, baseline_predictions), 3))

print("Validation balanced accuracy:",
      round(balanced_accuracy_score(y_validation, baseline_predictions), 3))

print("Inactivity precision:",
      round(precision_score(y_validation, baseline_predictions), 3))

print("Inactivity recall:",
      round(recall_score(y_validation, baseline_predictions), 3))

display(pd.DataFrame(
    confusion_matrix(y_validation, baseline_predictions, labels=[0, 1]),
    index=["Actual purchased again", "Actual inactive"],
    columns=["Predicted purchased again", "Predicted inactive"],
))

Validation accuracy: 0.499
Validation balanced accuracy: 0.5
Inactivity precision: 0.499
Inactivity recall: 1.0


,Predicted purchased again,Predicted inactive
Actual purchased again,0,1331
Actual inactive,0,1328


In [15]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=2000))
])

model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](8,)","['recency_days','order_count','purchase_value',...,'orders_last_90_days', 'purchase_value_last_90_days','orders_previous_90_days']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,8
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [17]:
validation_probabilities = model.predict_proba(X_validation)[:, 1]

validation_predictions = (
    validation_probabilities >= 0.5
).astype(int)

In [18]:
from sklearn.metrics import roc_auc_score

comparison = pd.DataFrame({
    "dummy": {
        "accuracy": accuracy_score(y_validation, baseline_predictions),
        "balanced_accuracy": balanced_accuracy_score(
            y_validation, baseline_predictions
        ),
        "precision": precision_score(
            y_validation, baseline_predictions, zero_division=0
        ),
        "recall": recall_score(
            y_validation, baseline_predictions, zero_division=0
        ),
    },
    "logistic_regression": {
        "accuracy": accuracy_score(y_validation, validation_predictions),
        "balanced_accuracy": balanced_accuracy_score(
            y_validation, validation_predictions
        ),
        "precision": precision_score(
            y_validation, validation_predictions, zero_division=0
        ),
        "recall": recall_score(
            y_validation, validation_predictions, zero_division=0
        ),
    },
}).T

display(comparison.round(3))

print(
    "Logistic regression ROC AUC:",
    round(roc_auc_score(y_validation, validation_probabilities), 3),
)

,accuracy,balanced_accuracy,precision,recall
dummy,0.499,0.500,0.499,1.000
logistic_regression,0.687,0.688,0.667,0.747


Logistic regression ROC AUC: 0.754


In [19]:
display(pd.DataFrame(
    confusion_matrix(
        y_validation,
        validation_predictions,
        labels=[0, 1],
    ),
    index=["Actual purchased again", "Actual inactive"],
    columns=["Predicted purchased again", "Predicted inactive"],
))

,Predicted purchased again,Predicted inactive
Actual purchased again,836,495
Actual inactive,336,992


In [20]:
validation_results = validation[
    ["customer_id", "recency_days"]
].copy()

validation_results["actual_inactive"] = y_validation.to_numpy()
validation_results["predicted_risk"] = validation_probabilities

In [21]:
import math

n_target = math.ceil(len(validation_results) * 0.20)
overall_inactivity_rate = validation_results["actual_inactive"].mean()

rankings = {
    "logistic_regression": "predicted_risk",
    "recency_rule": "recency_days",
}

results = []

for method, ranking_column in rankings.items():
    selected = (
        validation_results
        .sort_values(
            [ranking_column, "customer_id"],
            ascending=[False, True],
        )
        .head(n_target)
    )

    inactive_found = selected["actual_inactive"].sum()
    precision_at_20 = selected["actual_inactive"].mean()
    recall_at_20 = (
        inactive_found / validation_results["actual_inactive"].sum()
    )

    results.append({
        "method": method,
        "customers_selected": len(selected),
        "inactive_found": inactive_found,
        "precision_at_20": precision_at_20,
        "recall_at_20": recall_at_20,
        "lift_at_20": precision_at_20 / overall_inactivity_rate,
    })

display(pd.DataFrame(results).set_index("method").round(3))

,customers_selected,inactive_found,precision_at_20,recall_at_20,lift_at_20
method,,,,,
logistic_regression,532,403,0.758,0.303,1.517
recency_rule,532,370,0.695,0.279,1.393


In [22]:
coefficients = pd.DataFrame({
    "feature": feature_columns,
    "coefficient": model.named_steps["classifier"].coef_[0],
})

coefficients["absolute_coefficient"] = (
    coefficients["coefficient"].abs()
)

display(
    coefficients
    .sort_values("absolute_coefficient", ascending=False)
    .drop(columns="absolute_coefficient")
    .round(3)
)

,feature,coefficient
1,order_count,-1.046
5,orders_last_90_days,-0.484
7,orders_previous_90_days,-0.389
0,recency_days,0.315
4,observed_tenure_days,-0.213
3,avg_order_value,-0.141
6,purchase_value_last_90_days,0.082
2,purchase_value,-0.005


In [23]:
assert np.isfinite(X_test.to_numpy()).all()

test_probabilities = model.predict_proba(X_test)[:, 1]
test_predictions = (test_probabilities >= 0.5).astype(int)

In [24]:
print("Test accuracy:",
      round(accuracy_score(y_test, test_predictions), 3))

print("Test balanced accuracy:",
      round(balanced_accuracy_score(y_test, test_predictions), 3))

print("Test precision:",
      round(precision_score(y_test, test_predictions, zero_division=0), 3))

print("Test recall:",
      round(recall_score(y_test, test_predictions, zero_division=0), 3))

print("Test ROC AUC:",
      round(roc_auc_score(y_test, test_probabilities), 3))

display(pd.DataFrame(
    confusion_matrix(y_test, test_predictions, labels=[0, 1]),
    index=["Actual purchased again", "Actual inactive"],
    columns=["Predicted purchased again", "Predicted inactive"],
))

Test accuracy: 0.65
Test balanced accuracy: 0.663
Test precision: 0.541
Test recall: 0.725
Test ROC AUC: 0.73


,Predicted purchased again,Predicted inactive
Actual purchased again,1012,671
Actual inactive,299,790


In [25]:
test_results = test[["customer_id", "recency_days"]].copy()
test_results["actual_inactive"] = y_test.to_numpy()
test_results["predicted_risk"] = test_probabilities

n_target = math.ceil(len(test_results) * 0.20)
overall_rate = test_results["actual_inactive"].mean()

results = []

for method, ranking_column in rankings.items():
    selected = (
        test_results
        .sort_values(
            [ranking_column, "customer_id"],
            ascending=[False, True],
        )
        .head(n_target)
    )

    precision = selected["actual_inactive"].mean()

    results.append({
        "method": method,
        "customers_selected": len(selected),
        "inactive_found": selected["actual_inactive"].sum(),
        "precision_at_20": precision,
        "recall_at_20": (
            selected["actual_inactive"].sum()
            / test_results["actual_inactive"].sum()
        ),
        "lift_at_20": precision / overall_rate,
    })

display(pd.DataFrame(results).set_index("method").round(3))

,customers_selected,inactive_found,precision_at_20,recall_at_20,lift_at_20
method,,,,,
logistic_regression,555,366,0.659,0.336,1.679
recency_rule,555,343,0.618,0.315,1.573
